In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import akshare as ak
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [6]:
import ssl
import warnings
import urllib3
import requests

In [7]:
# 1. Silenciar las advertencias visuales de conexiones no verificadas
warnings.filterwarnings("ignore")
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. Desactivar verificación en el motor SSL nativo de Python
ssl._create_default_https_context = ssl._create_unverified_context

# 3. Desactivar verificación en requests (corrige FRED, pandas_datareader y akshare)
_old_session_init = requests.Session.__init__
def _insecure_session_init(self, *args, **kwargs):
    _old_session_init(self, *args, **kwargs)
    self.verify = False
requests.Session.__init__ = _insecure_session_init

# 4. Desactivar verificación en curl_cffi (corrige el motor interno de yfinance)
try:
    from curl_cffi import requests as cureq
    _old_cureq_init = cureq.Session.__init__
    def _insecure_cureq_init(self, *args, **kwargs):
        _old_cureq_init(self, *args, **kwargs)
        self.verify = False
    cureq.Session.__init__ = _insecure_cureq_init
except ImportError:
    pass

print("Parche SSL aplicado con éxito. Conexiones liberadas.")

Parche SSL aplicado con éxito. Conexiones liberadas.


In [14]:
# ==============================================================================
# 1. PARÁMETROS TEMPORALES Y TICKERS
# ==============================================================================
fecha_inicio = "2020-01-01"
fecha_fin = "2026-06-01"
ONZAS_POR_GRAMO = 31.1034768

print("1. Descargando mercados de oro occidentales y fundamentales macro...")

# Activos de Oro: GC=F (COMEX), GLD (Proxy LBMA), 518880.SS (ETF China)
# Fundamentales: UUP (Proxy Dólar / DXY), TIP (Tasas Reales TIPS), CNY=X (Tipo de cambio)
tickers_yf = ["GC=F", "GLD", "518880.SS", "UUP", "TIP", "CNY=X"]
df_yf = yf.download(tickers_yf, start=fecha_inicio, end=fecha_fin, auto_adjust=False)['Adj Close']

# Normalizar índice temporal para eliminar zonas horarias
df_yf.index = pd.to_datetime(df_yf.index).tz_localize(None).normalize()

# Separación de series individuales
comex = df_yf['GC=F'].rename("COMEX_Futuros_USD")
usdcny = df_yf['CNY=X'].rename("USD_CNY")
dolar_dxy = df_yf['UUP'].rename("Dolar_DXY")
tasas_reales = df_yf['TIP'].rename("Tasas_Reales_TIPS")

# Reconstruir precio físico de Londres mediante el proxy GLD
ratio_conversion_gld = (comex / df_yf['GLD']).median()
lbma_proxy_usd = (df_yf['GLD'] * ratio_conversion_gld).rename("LBMA_Proxy_Spot_USD")

# ==============================================================================
# 2. DESCARGA DE MERCADOS CHINOS (akshare)
# ==============================================================================
print("2. Descargando series domésticas de China (SGE y SHFE)...")

# Spot SGE (Au99.99) en RMB/g
df_sge = ak.spot_hist_sge(symbol="Au99.99")
df_sge['date'] = pd.to_datetime(df_sge['date']).dt.tz_localize(None).dt.normalize()
sge_rmb = df_sge.set_index('date')['close'].loc[fecha_inicio:fecha_fin].rename("SGE_Au9999_RMB_g")

# Futuros SHFE (au0) en RMB/g
df_shfe = ak.futures_zh_daily_sina(symbol="au0")
df_shfe['date'] = pd.to_datetime(df_shfe['date']).dt.tz_localize(None).dt.normalize()
shfe_rmb = df_shfe.set_index('date')['close'].loc[fecha_inicio:fecha_fin].rename("SHFE_Futuros_RMB_g")

# ==============================================================================
# 3. ALINEACIÓN, CONVERSIÓN Y CALENDARIOS CRUZADOS
# ==============================================================================
print("3. Homogenizando unidades a USD/oz y sincronizando panel...")
panel_precios = pd.concat([comex, lbma_proxy_usd, sge_rmb, shfe_rmb, usdcny, dolar_dxy, tasas_reales], axis=1)

# Normalizar precios de China a USD por onza troy
panel_precios['SGE_Spot_USD_oz'] = (panel_precios['SGE_Au9999_RMB_g'] * ONZAS_POR_GRAMO) / panel_precios['USD_CNY']
panel_precios['SHFE_Futuros_USD_oz'] = (panel_precios['SHFE_Futuros_RMB_g'] * ONZAS_POR_GRAMO) / panel_precios['USD_CNY']

# Relleno hacia adelante para festivos asincrónicos
panel_precios = panel_precios.ffill().dropna()

# ==============================================================================
# 4. RETORNOS LOGARÍTMICOS CONTINUOS
# ==============================================================================
# Seleccionar series de precios finales y factores macro
series_analisis = [
    'COMEX_Futuros_USD', 'LBMA_Proxy_Spot_USD', 
    'SGE_Spot_USD_oz', 'SHFE_Futuros_USD_oz',
    'Dolar_DXY', 'Tasas_Reales_TIPS'
]

# r_t = ln(P_t / P_{t-1})
retornos = np.log(panel_precios[series_analisis] / panel_precios[series_analisis].shift(1)).dropna()

# ==============================================================================
# 5. FILTRADO ECONOMÉTRICO OLS MULTI-MERCADO (EXTRACCIÓN DE RESIDUOS)
# ==============================================================================
print("\n4. Estimando regresiones de factores comunes por mercado...")

# Matriz de variables exógenas fundamentales
X = retornos[['Dolar_DXY', 'Tasas_Reales_TIPS']]
X = sm.add_constant(X)

mercados_oro = ['COMEX_Futuros_USD', 'LBMA_Proxy_Spot_USD', 'SGE_Spot_USD_oz', 'SHFE_Futuros_USD_oz']
df_residuos = pd.DataFrame(index=retornos.index)
resumen_modelos = {}

print("-" * 65)
print(f"{'Mercado':<22} | {'Beta Dólar':<12} | {'Beta TIPS':<12} | {'R²':<8}")
print("-" * 65)

for mercado in mercados_oro:
    y = retornos[mercado]
    modelo = sm.OLS(y, X).fit()
    
    # Guardar la serie del residuo puro (\epsilon_t)
    nombre_residuo = f"Residuo_{mercado}"
    df_residuos[nombre_residuo] = modelo.resid
    
    # Resumen para diagnóstico
    b_const, b_dolar, b_tips = modelo.params
    r2 = modelo.rsquared
    print(f"{mercado:<22} | {b_dolar:>11.4f} | {b_tips:>11.4f} | {r2:>7.2%}")

print("-" * 65)

# ==============================================================================
# 6. VARIABLES ESTRUCTURALES Y CONSOLIDACIÓN FINAL
# ==============================================================================
# Primas y Spreads en niveles de precios
panel_precios['Shanghai_Premium_USD'] = panel_precios['SGE_Spot_USD_oz'] - panel_precios['LBMA_Proxy_Spot_USD']
panel_precios['Spread_COMEX_LBMA_USD'] = panel_precios['COMEX_Futuros_USD'] - panel_precios['LBMA_Proxy_Spot_USD']

# Spread puramente anómalo (diferencial de residuos entre Asia y Occidente)
df_residuos['Residuo_Spread_SGE_LBMA'] = df_residuos['Residuo_SGE_Spot_USD_oz'] - df_residuos['Residuo_LBMA_Proxy_Spot_USD']

# Unir retornos brutos, fundamentales y residuos anómalos
panel_econometrico = pd.concat([retornos, df_residuos], axis=1)

# Guardar matrices procesadas para modelación en R
panel_precios.to_csv("../data/panel_precios_niveles.csv")
panel_econometrico.to_csv("../data/panel_retornos_y_residuos.csv")

print("\nArchivos exportados exitosamente:")
print("1. data/processed/panel_precios_niveles.csv (Niveles, primas y spreads en USD)")
print("2. data/processed/panel_retornos_y_residuos.csv (Retornos y residuos e_t para R)")

[**********************50%                       ]  3 of 6 completed

1. Descargando mercados de oro occidentales y fundamentales macro...


[*********************100%***********************]  6 of 6 completed


2. Descargando series domésticas de China (SGE y SHFE)...
3. Homogenizando unidades a USD/oz y sincronizando panel...

4. Estimando regresiones de factores comunes por mercado...
-----------------------------------------------------------------
Mercado                | Beta Dólar   | Beta TIPS    | R²      
-----------------------------------------------------------------
COMEX_Futuros_USD      |     -0.7835 |      0.5074 |  15.92%
LBMA_Proxy_Spot_USD    |     -0.8907 |      0.5910 |  22.30%
SGE_Spot_USD_oz        |     -0.2005 |      0.1242 |   1.08%
SHFE_Futuros_USD_oz    |     -0.2152 |      0.1250 |   1.15%
-----------------------------------------------------------------

Archivos exportados exitosamente:
1. data/processed/panel_precios_niveles.csv (Niveles, primas y spreads en USD)
2. data/processed/panel_retornos_y_residuos.csv (Retornos y residuos e_t para R)
